# 🫀 실험 22-A — **축별 전이 강건성을 같은 자로 잰다**

**MedKOS / `notebooks/exp22a_axis_transfer.ipynb`** · 퀘스트 `ailab-2026-0015`

> 먼저 읽을 것: **`pipelines/SCORING_RULES.md`** (R1~R9)

## 이 실험이 하는 일 — "새로 재기" 가 아니라 "같은 자로 모으기"

축별 전이 결과가 이미 **두 군데 흩어져 있다**:

| 축 | 어디 | 지표 | 문제 |
|---|---|---|---|
| 형태(MI) | 실험20 / 20c / 20d | **AUROC** 낙폭 | — |
| 비트(S·V) | `mit-bih/PAPER.md §6.5` | **PR-AUC** + lift | 코호트 간 비교 불가 |

`PR-AUC` 는 기저율에 붙어 있고, `lift` 는 **천장이 `1/기저율`** 이라 코호트마다 다르다
(MIT-BIH DS2 26.8× vs INCART 89.3× — 3배 차이). **R4 위반**이다.

→ **AUROC 로 다시 뽑아** 축 간 낙폭을 같은 자로 잰다.

## 그리고 §6.5 에는 짝이 없다

§6.5 는 **cross(INCART) 만** 있고 **within 기준선이 같은 모델에서 나오지 않았다**.
낙폭 = `within − cross` 이므로 **같은 학습으로 둘 다** 예측해야 한다.

> 그래서 이 실험은 `colab_crossdb.py` 의 학습 분할을 **MIT-BIH 전체 → DS1** 로 바꾼다.
> 그러면 `DS2`(within)와 `INCART`(cross)를 **같은 모델**로 예측할 수 있다.
> 이건 §6.5 의 **한계 L3**(PID 를 레코드로 셈)도 부분적으로 고친다.

## 사전등록

| 관문 | 내용 | 지지 조건 |
|---|---|---|
| **G0** | 자산·분할 점검 — `pid` 배열 존재 · 환자 vs 레코드 수 · DS1/DS2 누수 0 | 하나라도 실패하면 **중단** |
| **G1** | Drive pkl 10개에서 V/S Δ 재계산 → 미검증 인용을 실측으로 대체 | 재현되면 인용 해금 |
| **P-A★★** | **`V`(형태 정의 비트)의 낙폭 < `S`(타이밍 정의 비트)의 낙폭** | 차 > 0.05 (AUROC) |
| **P-B★★** | 리듬 축을 넣으면 `S` 의 **낙폭이 줄어드나** | `S` 낙폭(v2) < `S` 낙폭(v1) |
| **P-C** | BN 적응이 낙폭을 복구하나 | 복구되면 분포 shift · 안 되면 **판별 축 부재** |
| **P-D** | 축 비교표 — 형태(MI) 낙폭 vs 비트 V·S 낙폭 | 서술 보고(문턱 없음) |

### P-A 가 왜 핵심인가

`V`(심실조기박동)는 **넓은 QRS** 라는 형태로 정의된다 — 기기·인구가 바뀌어도 형태는 형태다.
`S`(심방조기박동)는 **"그 환자 평소보다 이르다"** 는 개인 기준 상대량이다 — 기저 심박수가
다른 인구로 옮기면 기준선이 통째로 이동한다.

**같은 실행 · 같은 모델 · 같은 테스트셋** 안에서 두 클래스를 비교하므로
`V` 가 **양성 대조군** 역할을 한다 — "실험이 망가진 게 아니라 `S` 만 다르게 행동한다".

## 하지 않는 것

- 새 백본 · 하이퍼파라미터 튜닝 · 임계값 조정
- `mit-bih/PAPER.md §6.5` 의 기존 수치 **수정** (병기만 한다)
- 비트 단위와 레코드 단위 **성능의 직접 비교** — 낙폭만 비교한다


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/SCORING_RULES.md R2·R3)
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    """사전등록 관문의 유일한 계약: 지지 / 기각 / **미결**."""
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return True
        if hi < thr: return False
    else:
        if hi < thr: return True
        if lo > thr: return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def t_ci(v, conf=.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2: return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

def t_ci_logit(v, conf=.95):
    """R2 — 유계 지표(0~1)는 logit 에서. 실험20b 가 특이도 CI 를 음수로 냈다."""
    v = np.clip(np.asarray([x for x in v if np.isfinite(x)], float), 1e-6, 1 - 1e-6)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan), np.nan, np.nan
    m, lo, hi = t_ci(np.log(v / (1 - v)), conf)
    f = lambda x: float(1 / (1 + np.exp(-x)))
    return f(m), f(lo), f(hi)

def boot_indices(n, B, seed):
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

class AssetError(RuntimeError):
    """자산·분할이 가정과 다르면 **추측하지 않고 멈춘다**."""

print("사전점검 적재: decide · t_ci · t_ci_logit · boot_indices · AssetError")


In [ ]:
# CELL 1 — 설정
# ★ WST(colab_step12_wst.py) 가 kymatio 를 쓴다. **여기서 미리** 깔아야 한다.
#   colab_step12_wst.py 안에도 자동설치가 있지만 설치 후 importlib.invalidate_caches()
#   를 안 불러서, 같은 세션에서 한 번 실패한 import 는 캐시 때문에 계속 실패한다
#   (같은 프로젝트의 colab_bootstrap.py::_ensure 는 그걸 고쳐놨다 — step12 만 빠졌다).
!pip -q install kymatio

import os, sys, json, time, pickle, subprocess, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")          # colab_crossdb.py 의 _BASE
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SMOKE = False        # ★ True 로 두면 시드 1개만 — 학습 루프가 도는지 먼저 확인한다
SEEDS = [2000] if SMOKE else [2000, 2001, 2002, 2003, 2004]   # crossdb 기본 시드대
KWST, BOOT, SEED0 = 40, 2000, 20260801
GAP_THR = 0.05                               # P-A 문턱 (AUROC 낙폭 차)

# ── 형태 축(MI) 낙폭 — 실험20/20d 실측. **비교 상대**로만 쓴다
MI_DROPS = {"IMI": 0.1246, "ILMI": 0.0877, "ASMI": 0.0908}
MI_NOTE = ("실험20 실측(v2-narrow 채점 3부위). AMI 는 라벨 오류였으므로 제외 "
           "— 실험20c·20d 참조. 내부는 5겹 OOF·외부는 전량 학습이라 **과소추정**")

# ── Drive 자산 (mit-bih/ASSETS_DRIVE.md 에 등록된 것)
PKL_DIR_HINT = "1ZSiLz1xoT-8aHPjE3yc4TBHxOYor2vD2"   # 확률 pkl 10개
PKL_NAMES = [f"mit_only_{1000+i}.pkl" for i in range(5)] + \
            [f"mit_svdb_{1000+i}.pkl" for i in range(5)]

CONFIG = dict(exp="exp22a_axis_transfer", quest="ailab-2026-0015",
              parent_exp=["exp20_ptbdb", "exp20d_combined", "mit-bih/PAPER.md §6.5"],
              purpose=("축별 전이 강건성을 **같은 자(AUROC)** 로 모은다. §6.5 는 PR-AUC 라 "
                       "코호트 간 비교가 안 되고, within 기준선이 같은 모델에서 안 나왔다"),
              dataset="MIT-BIH Arrhythmia (DS1 학습 / DS2 within) + INCART (cross)",
              change_one_thing=("colab_crossdb.py 의 학습 분할을 MIT-BIH 전체 → DS1 로 바꾼다. "
                                "백본·특징·하이퍼파라미터는 그대로"),
              seeds=SEEDS, kwst=KWST, gap_thr=GAP_THR,
              mi_drops=MI_DROPS, mi_note=MI_NOTE,
              metric_rule=("R4 — 코호트를 가로지르는 비교는 **AUROC** 로만 한다. "
                           "PR-AUC 는 기저율에 붙고 lift 는 천장이 1/기저율 이라 "
                           "MIT-BIH DS2(26.8x)와 INCART(89.3x)가 3배 다르다"),
              predictions={
                  "G0": "pid 배열 존재 · 환자 vs 레코드 수 보고 · DS1/DS2 누수 0",
                  "G1": "Drive pkl 10개에서 V/S Δ 재계산 → 미검증 인용 해금",
                  "P-A": f"V 낙폭 < S 낙폭, 차 > {GAP_THR} (AUROC)",
                  "P-B": "리듬 축을 넣으면 S 의 낙폭이 줄어든다",
                  "P-C": "BN 적응이 낙폭을 복구하나 (복구 실패 = 판별 축 부재)",
                  "P-D": "축 비교표 — 형태(MI) vs 비트 V·S. 서술 보고"},
              caveat=("비트 단위와 레코드 단위의 **성능**은 비교하지 않는다 — 낙폭만 비교한다. "
                      "§6.5 의 한계 L1·L2·L4·L5 는 그대로 남는다(L3 만 부분 교정)"))
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp22a_axis", CONFIG, project=PROJECT)
run.log(f"mit-bih 자산 루트: {MITBIH}")
run.log(f"형태 축 비교 상대(실험20): {MI_DROPS}")


In [ ]:
# CELL 2 — 【G0】 자산·분할 점검. **가정이 틀리면 여기서 멈춘다**
#   실험20 의 교훈: 무거운 계산 전에 값싼 관문으로 죽인다.
need = {"mamba_data.npz": "MIT-BIH 비트·라벨·pid",
        "incart_data.npz": "INCART 비트·라벨·pid",
        "colab_crossdb.py": "교차DB 하니스",
        "colab_step12_wst.py": "WST 추출기"}
miss = [f for f in need if not os.path.exists(os.path.join(MITBIH, f))]
if miss:
    raise AssetError(
        f"필요한 자산이 없다: {miss}\n"
        f"  찾은 곳: {MITBIH}\n"
        f"  → mit-bih/ASSETS_DRIVE.md 를 보고 위치를 확인할 것. **추측해서 만들지 않는다.**")
run.log("자산 확인 ✅ " + " · ".join(need))

# ── 패키지 게이트: 무거운 계산 전에 import 가 되는지 **실제로** 해본다
#   실험22-A 첫 실행에서 CELL 4 가 kymatio 없이 여기까지 와서 터졌다.
for _pkg, _why in (("kymatio", "WST 산란변환(colab_step12_wst.py)"),
                   ("torch", "백본"), ("sklearn", "스케일러·특징선택")):
    try:
        importlib.import_module(_pkg)
    except ModuleNotFoundError:
        raise AssetError(
            f"패키지 '{_pkg}' 를 import 할 수 없다 ({_why}).\n"
            f"  → CELL 1 의 `!pip -q install {_pkg}` 가 실패했거나 런타임 재시작이 필요하다.\n"
            "     학습으로 넘어가지 않는다.")
run.log("패키지 확인 ✅ kymatio · torch · sklearn")

_DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
_DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
assert not (set(_DS1) & set(_DS2)), "DS1/DS2 가 겹친다"

run.log("\n【G0】 npz 키 전수 — 가정한 키가 없으면 중단")
KEYS = {}
for f in ("mamba_data.npz", "incart_data.npz"):
    d = np.load(os.path.join(MITBIH, f))
    KEYS[f] = list(d.files)
    run.log(f"  {f:<20} {KEYS[f]}")
for f, want in (("mamba_data.npz", ("beat", "y", "pid", "feats")),
                ("incart_data.npz", ("beat", "y", "pid", "pre_rr", "post_rr"))):
    lack = [k for k in want if k not in KEYS[f]]
    if lack:
        raise AssetError(f"{f} 에 {lack} 이 없다 · 실제 키 {KEYS[f]}\n"
                         "  → 레코드 ID(pid) 없이는 환자 단위 분할을 할 수 없다. 중단한다.")
run.log("  ✅ 가정한 키가 전부 있다")

dm = np.load(os.path.join(MITBIH, "mamba_data.npz"))
di = np.load(os.path.join(MITBIH, "incart_data.npz"))
mpid, my = dm["pid"], dm["y"]; ipid, iy = di["pid"], di["y"]
TR = np.isin(mpid, _DS1); TE = np.isin(mpid, _DS2)
run.log(f"\n  MIT-BIH {len(my):,}비트 · 레코드 {len(np.unique(mpid))}개")
run.log(f"    DS1(학습) {int(TR.sum()):,}비트 / 레코드 {len(np.unique(mpid[TR]))}개")
run.log(f"    DS2(within) {int(TE.sum()):,}비트 / 레코드 {len(np.unique(mpid[TE]))}개")
run.log(f"    미분류 {int((~TR & ~TE).sum()):,}비트 (DS1·DS2 어디에도 없는 레코드)")
if TR.sum() == 0 or TE.sum() == 0:
    raise AssetError("DS1 또는 DS2 가 비었다 — pid 가 레코드 번호가 아닐 수 있다")
leak = set(np.unique(mpid[TR])) & set(np.unique(mpid[TE]))
if leak:
    raise AssetError(f"DS1/DS2 레코드 누수 {leak}")
run.log("  ✅ DS1/DS2 레코드 누수 0")

run.log(f"\n  INCART {len(iy):,}비트 · **pid 고유값 {len(np.unique(ipid))}개**")
run.log("    ⚠️ §6.5 한계 L3 — INCART 실제 환자는 32명인데 pid 를 레코드(75)로 셌다.")
run.log("       이 실험은 INCART 를 **테스트로만** 쓰고 내부 분할을 안 하므로 L3 의")
run.log("       영향은 '환자 단위 CI 를 못 낸다' 로 한정된다. 낙폭 점추정은 유효하다.")
for tag, y in (("MIT-BIH DS1", my[TR]), ("MIT-BIH DS2", my[TE]), ("INCART", iy)):
    n = len(y)
    run.log(f"    {tag:<14} N/S/V = {int((y==0).sum()):>6,}/{int((y==1).sum()):>5,}"
            f"/{int((y==2).sum()):>5,}  · S 기저율 {(y==1).mean():.4f} · "
            f"V 기저율 {(y==2).mean():.4f}")
run.log("\n  ※ 기저율이 코호트마다 다르다 → **PR-AUC·lift 로 비교하지 않는다**(R4)")
_s1, _s2 = float((my[TR] == 1).mean()), float((my[TE] == 1).mean())
run.log(f"  ⚠️ **DS1 S {_s1:.4f} vs DS2 S {_s2:.4f} — {_s2/_s1:.1f}배 차이.** MIT-BIH 고유 성질이다.")
run.log("     즉 'within' 기준선조차 순수한 동일분포 검정이 아니다(유병률 이동이 이미 있다).")
run.log("     AUROC 는 유병률 무관이라 낙폭 계산은 유효하지만, **within 을 '이상적 상한'**")
run.log("     으로 읽으면 안 된다 — 낙폭은 그만큼 **과소추정**된다. 한계로 기록한다.")
CONFIG["within_prevalence_shift"] = {"DS1_S": _s1, "DS2_S": _s2, "ratio": _s2 / max(_s1, 1e-9)}
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【G1】 Drive pkl 에서 V/S Δ 재계산 — 미검증 인용을 실측으로 바꾼다
#   mit-bih/ASSETS_DRIVE.md 의 '인용 대기' 표를 여기서 해금하거나 폐기한다.
from sklearn.metrics import roc_auc_score, average_precision_score

def find_pkl(name):
    for root, _, files in os.walk(DRIVE_ROOT):
        if name in files:
            return os.path.join(root, name)
    return None

paths = {n: find_pkl(n) for n in PKL_NAMES}
found = {n: p for n, p in paths.items() if p}
run.log(f"\n【G1】 확률 pkl {len(found)}/{len(PKL_NAMES)}개 발견")
G1 = None
if len(found) < len(PKL_NAMES):
    run.log(f"  ⚠️ 없는 것: {[n for n in PKL_NAMES if n not in found]}")
    run.log("  → G1 **건너뛴다**. ASSETS_DRIVE.md 의 인용 대기 표는 계속 잠긴 상태로 둔다")
    run.log("     (이 실험의 주가설 P-A~P-D 는 G1 없이도 성립한다)")
else:
    run.log(f"  경로 예: {list(found.values())[0]}")
    obj = pickle.load(open(list(found.values())[0], "rb"))
    ks = list(obj.keys()) if isinstance(obj, dict) else type(obj).__name__
    run.log(f"  pkl 구조: {ks}")
    if not (isinstance(obj, dict) and "proba" in obj):
        run.log("  ⚠️ 'proba' 키가 없다 — 구조가 기대와 다르다. G1 건너뛴다(추측 금지)")
    else:
        # 대조군(mit_only) vs 실험군(mit_svdb) 를 시드별로 짝지어 비교
        yv = my[TE] if len(obj["proba"]) == int(TE.sum()) else None
        if yv is None:
            run.log(f"  ⚠️ proba 길이 {len(obj['proba'])} != DS2 {int(TE.sum())} — 테스트셋이"
                    " 다르다. G1 건너뛴다")
        else:
            G1 = {}
            for arm in ("mit_only", "mit_svdb"):
                A = {"S": [], "V": []}
                for i in range(5):
                    o = pickle.load(open(found[f"{arm}_{1000+i}.pkl"], "rb"))
                    p = np.asarray(o["proba"], dtype="float64")
                    A["S"].append(roc_auc_score((yv == 1).astype(int), p[:, 1]))
                    A["V"].append(roc_auc_score((yv == 2).astype(int), p[:, 2]))
                G1[arm] = A
            run.log(f"  {'클래스':<6}{'mit_only':>12}{'mit_svdb':>12}{'Δ':>12}{'95% CI':>24}")
            for c in ("S", "V"):
                d = [G1["mit_svdb"][c][i] - G1["mit_only"][c][i] for i in range(5)]
                m, lo, hi = t_ci(d)
                run.log(f"  {c:<6}{np.mean(G1['mit_only'][c]):>12.4f}"
                        f"{np.mean(G1['mit_svdb'][c]):>12.4f}{m:>+12.4f}"
                        f"   [{lo:+.4f}, {hi:+.4f}]")
            run.log("  ※ 이건 **데이터 증강**(DS1+SVDB → DS2) 이지 교차DB 낙폭이 아니다.")
            run.log("     '축별 상반 반응' 으로 명명한다 — V 는 오르고 S 는 내리는가?")


In [ ]:
# CELL 4 — DS1 학습 → DS2(within) + INCART(cross) **같은 모델로** 예측
#   ★ colab_crossdb.py 의 함수를 그대로 쓰되 **학습 분할만** 바꾼다.
#     백본·특징·하이퍼파라미터는 손대지 않는다(그래야 §6.5 와 이어붙는다).
import torch
# ★ 두 파일을 **순서대로** 읽는다. split 은 crossdb 의 module-level 심볼에 의존한다.
for _f in ("colab_crossdb.py", "colab_crossdb_split.py"):
    _p = os.path.join(MITBIH, _f)
    if not os.path.exists(_p):
        raise AssetError(f"{_f} 가 없다: {_p}\n"
                         "  → colab_crossdb_split.py 는 실험22-A 에서 추가된 파일이다. "
                         "repo mit-bih/ 에서 Drive 로 올려야 한다")
    exec(open(_p).read(), globals())
globals()["_BASE"] = MITBIH
globals()["_FEATDIR"] = f"{MITBIH}/synergy_feats"

# split 이 쓰는 심볼이 전부 들어왔는지 확인 — 없으면 학습 전에 멈춘다
_need = ["_DS1", "_DS2", "_determinism", "_znorm", "_medref", "_net",
         "_incart_feats", "_crossdb_rhythm", "auto_weights", "run_crossdb_split"]
_lack = [n for n in _need if n not in globals()]
if _lack:
    raise AssetError(f"exec 후에도 {_lack} 이 없다.\n"
                     "  → Drive 의 colab_crossdb.py 가 repo 사본과 다를 수 있다. "
                     "추측하지 않고 멈춘다.")
run.log("하니스 적재 ✅ colab_crossdb.py + colab_crossdb_split.py")

# ★ 캐시 이름에 시드 수를 박는다. 안 그러면 스모크(1시드) 결과가 본 실행(5시드)을
#   가로채고, CELL 5 는 그걸 모른 채 채점한다.
CACHE = run.data(f"exp22a_probs_s{len(SEEDS)}_v1.npz")
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    P = {k: z[k] for k in z.files}
    run.log(f"\n예측 캐시 적중 {CACHE} · {list(P)}")
else:
    run.log("\n학습 시작 — DS1 만으로 학습해 DS2·INCART 를 같은 모델로 예측한다")
    t0 = time.time()
    P = run_crossdb_split(seeds=SEEDS, Kwst=KWST, train_mask=TR, test_mask=TE)
    np.savez_compressed(CACHE, **P)
    run.log(f"완료 {time.time()-t0:.0f}s → 캐시 저장")

# P 는 {"<구성>_<within|cross>_<raw|bn>": (시드, 비트, 3)} 형태여야 한다
run.log("\n예측 배열")
for k in sorted(P):
    run.log(f"  {k:<28}{np.shape(P[k])}")
EXPECT = [f"{c}_{s}_raw" for c in ("v1", "v2") for s in ("within", "cross")] + \
         ["y_within", "y_cross"]
lack = [k for k in EXPECT if k not in P]
if lack:
    raise AssetError(f"예측 배열에 {lack} 이 없다 · 있는 것 {sorted(P)}\n"
                     "  → run_crossdb_split 이 기대한 형식을 안 냈다. 채점으로 넘어가지 않는다.")


In [ ]:
# CELL 5 — 【P-A·P-B·P-C】 AUROC 낙폭 (R4 — 코호트를 가로지르므로 AUROC 만)
CLS = {"S": 1, "V": 2}
# ★ 라벨은 예측과 **같은 출처**에서 받는다(정렬이 어긋나면 조용히 다른 실험이 된다)
Y = {"within": np.asarray(P["y_within"]), "cross": np.asarray(P["y_cross"])}
for sp in ("within", "cross"):
    n_p, n_y = np.shape(P[f"v1_{sp}_raw"])[1], len(Y[sp])
    if n_p != n_y:
        raise AssetError(f"{sp}: 예측 {n_p}행 vs 라벨 {n_y}행 — 정렬이 안 맞는다")
run.log(f"\n라벨 정렬 확인 ✅ within {len(Y['within']):,} · cross {len(Y['cross']):,}")

def auc_(y, s, k):
    yy = (y == k).astype(int)
    return float(roc_auc_score(yy, s)) if yy.any() and (yy == 0).any() else np.nan

AUC, DROP = {}, {}
for cfg in ("v1", "v2"):
    for mode in ("raw", "bn"):
        key = f"{cfg}_{{}}_{mode}"
        if key.format("within") not in P or key.format("cross") not in P:
            continue
        for c, k in CLS.items():
            w = [auc_(Y["within"], P[key.format("within")][i][:, k], k)
                 for i in range(len(SEEDS))]
            x = [auc_(Y["cross"], P[key.format("cross")][i][:, k], k)
                 for i in range(len(SEEDS))]
            AUC[(cfg, mode, c, "within")] = w
            AUC[(cfg, mode, c, "cross")] = x
            DROP[(cfg, mode, c)] = [w[i] - x[i] for i in range(len(w))]

run.log("\n" + "=" * 108)
run.log("【P-A·P-B·P-C】 AUROC — within(MIT-BIH DS2) vs cross(INCART) · 같은 학습")
run.log("=" * 108)
run.log(f"  {'구성':<6}{'적응':<6}{'클래스':<6}{'within':>10}{'cross':>10}{'낙폭':>10}{'95% CI':>22}")
for (cfg, mode, c), d in sorted(DROP.items()):
    mw, _, _ = t_ci_logit(AUC[(cfg, mode, c, "within")])
    mx, _, _ = t_ci_logit(AUC[(cfg, mode, c, "cross")])
    m, lo, hi = t_ci(d)
    run.log(f"  {cfg:<6}{mode:<6}{c:<6}{mw:>10.4f}{mx:>10.4f}{m:>+10.4f}"
            f"   [{lo:+.4f}, {hi:+.4f}]")

# P-A: V 낙폭 < S 낙폭 (리듬 없는 v1 에서 — 형태 축만 준 상태가 가장 깨끗한 대비)
gapA = [DROP[("v1", "raw", "S")][i] - DROP[("v1", "raw", "V")][i] for i in range(len(SEEDS))]
mA, lA, hA = t_ci(gapA)
run.log(f"\n  P-A  S 낙폭 − V 낙폭 (v1·raw) = {mA:+.4f} [{lA:+.4f}, {hA:+.4f}] "
        f"vs 문턱 {GAP_THR}")
run.log("       V 는 넓은 QRS 라는 **형태**로 정의되고 S 는 '평소보다 이르다' 는")
run.log("       **개인 기준 상대량**이다. 같은 실행 안에서 V 가 양성 대조군 역할을 한다")

# P-B: 리듬 축이 S 의 낙폭을 줄이나
gapB = None
if ("v2", "raw", "S") in DROP:
    gapB = [DROP[("v1", "raw", "S")][i] - DROP[("v2", "raw", "S")][i] for i in range(len(SEEDS))]
    mB, lB, hB = t_ci(gapB)
    run.log(f"\n  P-B  S 낙폭(v1) − S 낙폭(v2) = {mB:+.4f} [{lB:+.4f}, {hB:+.4f}]")
    run.log("       양수면 **리듬 축이 전이를 구해준다**(§6.5 의 1.8x → 7.9x 를 AUROC 로 재현)")

# P-C: BN 적응이 복구하나
gapC = None
if ("v2", "bn", "S") in DROP:
    gapC = [DROP[("v2", "raw", "S")][i] - DROP[("v2", "bn", "S")][i] for i in range(len(SEEDS))]
    mC, lC, hC = t_ci(gapC)
    run.log(f"\n  P-C  S 낙폭(raw) − S 낙폭(BN 적응) = {mC:+.4f} [{lC:+.4f}, {hC:+.4f}]")
    run.log("       0 근처면 **입력 분포 shift 가 아니라 판별 축의 문제**다(§6.5 결론 재현)")


In [ ]:
# CELL 6 — 【P-D】 축 비교표 · 사전등록 채점
run.log("\n" + "=" * 108)
run.log("【P-D】 축별 낙폭 — 같은 자(AUROC)로")
run.log("=" * 108)
run.log(f"  {'축':<26}{'과제':<16}{'낙폭(AUROC)':>14}{'출처':>22}")
for s, d in MI_DROPS.items():
    run.log(f"  {'형태 (레코드·12유도)':<26}{'MI ' + s:<16}{d:>+14.4f}{'실험20':>22}")
for c, nm in (("V", "심실조기박동"), ("S", "심방조기박동")):
    m, lo, hi = t_ci(DROP[("v1", "raw", c)])
    run.log(f"  {'비트 (형태특징만)':<26}{nm:<16}{m:>+14.4f}{'실험22-A':>22}")
if ("v2", "raw", "S") in DROP:
    for c, nm in (("V", "심실조기박동"), ("S", "심방조기박동")):
        m, _, _ = t_ci(DROP[("v2", "raw", c)])
        run.log(f"  {'비트 (형태+리듬)':<26}{nm:<16}{m:>+14.4f}{'실험22-A':>22}")
run.log(f"\n  ⚠️ {MI_NOTE}")
run.log("  ⚠️ 비트 단위와 레코드 단위의 **성능**은 비교하지 않는다 — 낙폭만 비교한다")

run.log("\n" + "=" * 108)
run.log("【사전등록 채점】")
run.log("=" * 108)
V = {}
V["P-A"] = decide(lA, hA, GAP_THR, ">")
run.log(f"\n  P-A S 낙폭 − V 낙폭 > {GAP_THR}")
run.log(f"      {mA:+.4f} [{lA:+.4f}, {hA:+.4f}] → {MARK[V['P-A']]}")
run.log("      지지 = **전이 강건성은 축에 따라 다르다.** 타이밍은 기기 불변이 아니고")
run.log("             형태는 기기 불변이다 — 우리 직관과 반대일 수 있다")
if gapB is not None:
    V["P-B"] = decide(lB, hB, 0.0, ">")
    run.log(f"\n  P-B 리듬 축이 S 의 낙폭을 줄이나 (> 0)")
    run.log(f"      {mB:+.4f} [{lB:+.4f}, {hB:+.4f}] → {MARK[V['P-B']]}")
if gapC is not None:
    V["P-C"] = decide(lC, hC, 0.0, ">")
    run.log(f"\n  P-C BN 적응이 복구하나 (> 0)")
    run.log(f"      {mC:+.4f} [{lC:+.4f}, {hC:+.4f}] → {MARK[V['P-C']]}")
    run.log("      **미결/기각이면 §6.5 의 '판별 축 부재' 결론이 AUROC 에서도 재현**된다")
run.log("\n" + "  ".join(f"{k}: {MARK[v]}" for k, v in sorted(V.items())))
run.log(f"  G1: {'재계산 완료' if G1 else '건너뜀(자산 미발견 또는 구조 불일치)'}")
run.log("  ⚠️ §6.5 의 한계 L1·L2·L4·L5 는 그대로 남는다. L3 만 부분 교정했다")
run.log("  ⚠️ 시드 5개(t 배수 2.776)")


In [ ]:
# CELL 7 — 그림 + 저장
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))

labs, vals, cols = [], [], []
for c, nm in (("V", "비트 V\n(형태 정의)"), ("S", "비트 S\n(타이밍 정의)")):
    labs.append(nm); vals.append(np.mean(DROP[("v1", "raw", c)])); cols.append("#2ca02c" if c == "V" else "#d62728")
for s, d in MI_DROPS.items():
    labs.append(f"MI {s}\n(형태·레코드)"); vals.append(d); cols.append("#999999")
ax[0].bar(range(len(vals)), vals, color=cols)
ax[0].set_xticks(range(len(vals))); ax[0].set_xticklabels(labs, fontsize=8)
ax[0].axhline(0, c="k", lw=.8); ax[0].set_ylabel("AUROC 낙폭 (within − cross)")
ax[0].set_title(f"축별 전이 낙폭 · P-A {MARK[V['P-A']]}")

if gapB is not None:
    xs = np.arange(2); w = 0.35
    ax[1].bar(xs - w/2, [np.mean(DROP[("v1", "raw", c)]) for c in ("V", "S")], w,
              label="v1 형태만", color="#bbbbbb")
    ax[1].bar(xs + w/2, [np.mean(DROP[("v2", "raw", c)]) for c in ("V", "S")], w,
              label="v2 +리듬", color="#1f77b4")
    ax[1].set_xticks(xs); ax[1].set_xticklabels(["V", "S"])
    ax[1].axhline(0, c="k", lw=.8); ax[1].legend(fontsize=8)
    ax[1].set_ylabel("AUROC 낙폭"); ax[1].set_title(f"리듬 축의 구조 효과 · P-B {MARK[V.get('P-B')]}")
plt.tight_layout(); run.save_fig("exp22a_axis_transfer", fig); plt.show()

res = {
    "week": 2, "exp_id": "exp22a_axis", "quest": "ailab-2026-0015",
    "step": "exp22a-axis-transfer", "split": "inter",
    "task": "축별 전이 강건성을 같은 자(AUROC)로 모은다",
    "notebook": "notebooks/exp22a_axis_transfer.ipynb",
    "metric": "drop_gap_S_minus_V", "value": round(float(mA), 4),
    "passed": bool(V.get("P-A") is True),
    "seeds": SEEDS,
    "auroc": {f"{c}|{m}|{cl}|{sp}": float(np.mean(v)) for (c, m, cl, sp), v in AUC.items()},
    "drop": {f"{c}|{m}|{cl}": [float(x) for x in v] for (c, m, cl), v in DROP.items()},
    "mi_drops_reference": MI_DROPS, "mi_note": MI_NOTE,
    "g1_recomputed": (None if G1 is None else
                      {a: {c: [float(x) for x in G1[a][c]] for c in ("S", "V")} for a in G1}),
    "verdicts": {k: V[k] for k in V},
    "caveats": [
        "R4 — 코호트를 가로지르는 비교는 AUROC 로만. PR-AUC·lift 는 천장이 다르다",
        "비트 단위와 레코드 단위의 성능은 비교하지 않는다 — 낙폭만",
        "§6.5 의 한계 L1·L2·L4·L5 는 그대로. L3 만 부분 교정",
        "형태 축(MI) 낙폭은 내부 5겹 OOF vs 외부 전량 학습이라 과소추정",
        ("DS1 S 기저율 0.0187 vs DS2 0.0373 — within 기준선에도 이미 2배 유병률 이동이 "
         "있다. AUROC 는 무관하나 within 을 이상적 상한으로 읽으면 안 된다"),
        "G1 은 데이터 증강 실험이지 교차DB 낙폭이 아니다 — '축별 상반 반응'"],
}
run.save_json("result", res); run.finish(res)
print(json.dumps({k: res[k] for k in ("metric", "value", "verdicts")},
                 ensure_ascii=False, indent=2))
